In [21]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.tree import DecisionTreeClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import make_scorer, accuracy_score, f1_score
from sklearn import datasets
digits_dataset_X, digits_dataset_y = datasets.load_digits(return_X_y=True)
digits_df = pd.DataFrame(digits_dataset_X, columns=[f"pixel_{i}" for i in range(64)])
digits_df['label'] = digits_dataset_y

# Dataset paths
datasets = {
    "Digits": digits_df,
    "Rice": pd.read_csv("../../res/rice.csv"),
    "Credit Approval": pd.read_csv("../../res/credit_approval.csv"),
    "Parkinsons": pd.read_csv("../../res/parkinsons.csv")
}

def auto_preprocess(df):
    """Handles encoding and scaling for sklearn models."""
    df = df.copy()
    
    # Identify target (assuming it's the last column if not named 'label' or 'class')
    target_col = df.columns[-1]
    if 'label' in df.columns: target_col = 'label'
    if 'Diagnosis' in df.columns: target_col = 'Diagnosis'
    
    X = df.drop(columns=[target_col])
    y = df[target_col]

    # Encode target if categorical
    if y.dtype == 'object':
        y = LabelEncoder().fit_transform(y)

    # Encode features: Get dummies for categorical, scale numerical
    X = pd.get_dummies(X, drop_first=True)
    
    # Neural Nets are very sensitive to scaling; Decision Trees aren't, 
    # but scaling both ensures a fair comparison.
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    
    return X_scaled, y

# Define metrics to track
scoring = {
    'accuracy': 'accuracy',
    'f1_weighted': 'f1_weighted' # 'weighted' handles multi-class or imbalanced data well
}

results = []

for name, path in datasets.items():
    try:
        X, y = auto_preprocess(df)
        
        # Initialize Models
        models = {
            "Decision Tree": DecisionTreeClassifier(max_depth=9, min_samples_split=50),
        }
        
        print(f"\n--- Evaluating Dataset: {name} ---")
        
        for model_name, clf in models.items():
            skf = StratifiedKFold(n_splits=10, shuffle=True)
            cv_results = cross_validate(clf, X, y, cv=skf, scoring=scoring)
            
            mean_acc = np.mean(cv_results['test_accuracy'])
            mean_f1 = np.mean(cv_results['test_f1_weighted'])
            
            print(f"{model_name:>16} | Acc: {mean_acc:.4f} | F1: {mean_f1:.4f}")
            
            results.append({
                "Dataset": name,
                "Model": model_name,
                "Accuracy": mean_acc,
                "F1 Score": mean_f1
            })

    except FileNotFoundError:
        print(f"Skipping {name}: File not found at {path}")




--- Evaluating Dataset: Digits ---
   Decision Tree | Acc: 0.8208 | F1: 0.8167

--- Evaluating Dataset: Rice ---
   Decision Tree | Acc: 0.8358 | F1: 0.8228

--- Evaluating Dataset: Credit Approval ---
   Decision Tree | Acc: 0.7842 | F1: 0.7887

--- Evaluating Dataset: Parkinsons ---
   Decision Tree | Acc: 0.8305 | F1: 0.8258
